<a href="https://colab.research.google.com/github/Tanvi-diwakar/ML_projects/blob/main/cnn_jellyfish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JellyDrone image classification (CNN) - F21DL Group 11
Run the cells in order on a Colab **T4 GPU** (Runtime > Change runtime type). See `README.md` in this folder for the dataset link and results.
Run step 5 twice: once with `MODEL_TYPE = "resnet18"` and once with `"small_cnn"`.

In [2]:

!pip -q install gdown

import gdown
import zipfile
from pathlib import Path

FILE_ID = "19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0"

gdown.download(id=FILE_ID, output="/content/jellydrone_dataset.zip", quiet=False)

with zipfile.ZipFile("/content/jellydrone_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("/content/jellydrone_local")

from pathlib import Path
SRC = next(Path("/content/jellydrone_local").rglob("data.yaml")).parent

print("Dataset location:", SRC)


Downloading...
From (original): https://drive.google.com/uc?id=19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0
From (redirected): https://drive.google.com/uc?id=19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0&confirm=t&uuid=872f0d7e-e24d-4aa9-8ecf-09cb91555618
To: /content/jellydrone_dataset.zip
100%|██████████| 1.14G/1.14G [00:13<00:00, 83.6MB/s]


Dataset location: /content/jellydrone_local


In [3]:

import shutil
from collections import Counter
from pathlib import Path
import yaml

DST = Path("/content/jellydrone")
SPLITS = {"train": "train", "valid": "val", "test": "test"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

names = yaml.safe_load(open(SRC / "data.yaml"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
names = [str(n).strip().lower().replace(" ", "_") for n in names]

print("Classes in data.yaml:", names)

if DST.exists():
    shutil.rmtree(DST)

for rf_split, our_split in SPLITS.items():
    img_dir, lbl_dir = SRC / rf_split / "images", SRC / rf_split / "labels"
    counts, multi, no_label_file = Counter(), 0, 0

    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in IMG_EXT:
            continue

        lbl = lbl_dir / (img.stem + ".txt")
        area_per_class = Counter()

        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    cid, w, h = int(float(parts[0])), float(parts[3]), float(parts[4])
                    area_per_class[cid] += w * h
        else:
            no_label_file += 1

        if not area_per_class:
            cls = "no_jellyfish"
        else:
            if len(area_per_class) > 1:
                multi += 1
            cls = names[area_per_class.most_common(1)[0][0]]

        out = DST / our_split / cls
        out.mkdir(parents=True, exist_ok=True)
        shutil.copy(img, out / img.name)
        counts[cls] += 1

    print(f"\n{our_split}: {sum(counts.values())} images {dict(counts)}")
    print(f"   images with more than one class (largest box used): {multi}")
    print(f"   images with no label file (treated as no_jellyfish): {no_label_file}")

print("\nDone. Class folders written to", DST)


Classes in data.yaml: ['0', '1', '2']

train: 27432 images {'0': 4132, 'no_jellyfish': 16709, '1': 1617, '2': 4974}
   images with more than one class (largest box used): 717
   images with no label file (treated as no_jellyfish): 0

val: 2640 images {'no_jellyfish': 1598, '0': 389, '1': 171, '2': 482}
   images with more than one class (largest box used): 53
   images with no label file (treated as no_jellyfish): 0

test: 1236 images {'0': 171, 'no_jellyfish': 750, '1': 67, '2': 248}
   images with more than one class (largest box used): 35
   images with no label file (treated as no_jellyfish): 0

Done. Class folders written to /content/jellydrone


In [4]:

import os

for sp in ["train", "val", "test"]:
    row = {
        c: len(os.listdir(f"/content/jellydrone/{sp}/{c}"))
        for c in sorted(os.listdir(f"/content/jellydrone/{sp}"))
    }
    print(sp.ljust(6), row, "total", sum(row.values()))


train  {'0': 4132, '1': 1617, '2': 4974, 'no_jellyfish': 16709} total 27432
val    {'0': 389, '1': 171, '2': 482, 'no_jellyfish': 1598} total 2640
test   {'0': 171, '1': 67, '2': 248, 'no_jellyfish': 750} total 1236


In [5]:

from PIL import Image
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

SIZE = 256

def shrink(p):
    Image.open(p).convert("RGB").resize((SIZE, SIZE)).save(p, quality=95)

files = [
    p for p in Path("/content/jellydrone").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

with ThreadPoolExecutor(4) as ex:
    list(ex.map(shrink, files))

print(len(files), "images resized to", SIZE)


31308 images resized to 256


In [ ]:
# STEP 5 - Train on train/, pick best epoch on val/, report ONCE on test/.
# Run once with MODEL_TYPE = "resnet18" and once with "small_cnn".
# GenAI DISCLOSURE (F21DL group coursework, GenAI framework statement 6: code generation only)
# Tool: Claude (Anthropic). Summary of the prompts I (Shubham) gave Claude, in order:
# 1. I gave Claude the F21DL 2026/27 group coursework brief and our Group 11 pitch slides (jellyfish occurrence
#    prediction using JeDI occurrence records, Copernicus ocean data and JellyDrone aerial images). I explained that
#    our team has 5 people: 2 work on the tabular data (RQ1) and 3 work on the images (RQ2 / hypothesis H3: a
#    fine-tuned CNN outperforms classical image classifiers at telling Aurelia, Chrysaora and no-jellyfish water
#    apart). I said my part is the CNN code.
# 2. I asked: "Help me build a CNN for jellyfish image classification" with these constraints: PyTorch; runs on
#    Google Colab with a T4 GPU (nobody on the team has a GPU); a small CNN trained from scratch AND a fine-tuned
#    pretrained ResNet18 (coursework requirement R4); train/val/test folders with a fixed seed (42) so every
#    teammate's models are compared on the same split; augmentation on training images only; class weights for
#    class imbalance; keep the best checkpoint by validation accuracy; evaluate on the test set once and print
#    precision, recall, F1 and a confusion matrix; simple code with clear comments.
# 3. The dataset arrived as a Roboflow YOLOv8 object-detection export (images + bounding boxes), so I asked for a
#    converter that gives each image ONE label (class of the largest box; empty label file = no_jellyfish) and
#    keeps Roboflow's own train/valid/test split.
# 4. Training on the full-size photos was too slow on Colab's 2 CPU cores, so I asked how to speed it up and Claude
#    added a step that shrinks every image to 256x256.
# I ran all the code myself on the real data and checked the outputs.
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

# ---------------- settings you may change ----------------
DATA_DIR = "/content/jellydrone"   # <-- created by the converter cell above
MODEL_TYPE = "resnet18"      # "resnet18" (pretrained, fine-tuned) or "small_cnn" (built from scratch)
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 1e-3         # use 1e-4 for resnet18 if training is unstable
IMG_SIZE = 224
SEED = 42
# ---------------------------------------------------------

# Same seed = same results every run (needed so teammates can reproduce).
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- Data: augmentation only on training images ---
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet values
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", train_tf)
val_ds = datasets.ImageFolder(f"{DATA_DIR}/val", eval_tf)
test_ds = datasets.ImageFolder(f"{DATA_DIR}/test", eval_tf)
class_names = train_ds.classes
print("Classes:", class_names)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, BATCH_SIZE, num_workers=2)
test_dl = DataLoader(test_ds, BATCH_SIZE, num_workers=2)


# --- Models ---
def small_cnn(n_classes):
    """A simple CNN trained from scratch: 3 conv blocks then a classifier."""
    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                             nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(
        block(3, 32), block(32, 64), block(64, 128),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Dropout(0.5), nn.Linear(128, n_classes))


def resnet18_finetune(n_classes):
    """ResNet18 pretrained on ImageNet, last layer swapped for our classes."""
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m


model = (small_cnn if MODEL_TYPE == "small_cnn" else resnet18_finetune)(len(class_names)).to(device)

# Class weights help if one class has far fewer images than the others.
counts = np.bincount(train_ds.targets)
weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def run_epoch(dl, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


# --- Training: keep the checkpoint with the best validation accuracy ---
best_val = 0.0
ckpt = f"best_{MODEL_TYPE}.pt"
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, True)
    va_loss, va_acc = run_epoch(val_dl, False)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")
    if va_acc > best_val:
        best_val = va_acc
        torch.save(model.state_dict(), ckpt)

# --- Final test: run ONCE, on the best checkpoint ---
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()
preds, truth = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        truth += y.tolist()
print("\nTEST RESULTS for", MODEL_TYPE)
print(classification_report(truth, preds, target_names=class_names, digits=3))
print("Confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(truth, preds))


Using device: cpu
Classes: ['0', '1', '2', 'no_jellyfish']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 90.7MB/s]


In [ ]:
# STEP 6 - Save the trained models so they survive when Colab's temporary disk is wiped
!mkdir -p /content/drive/MyDrive/jellydrone_results
!cp /content/best_*.pt /content/drive/MyDrive/jellydrone_results/
!ls -la /content/drive/MyDrive/jellydrone_results/
